In [1]:
# Importing basic libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("../data/data2.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145230 entries, 0 to 145229
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   MinTemp            145230 non-null  float64
 1   MaxTemp            145230 non-null  float64
 2   Rainfall           145230 non-null  float64
 3   Evaporation        145230 non-null  float64
 4   WindGustSpeed      145230 non-null  float64
 5   RainTomorrow       145230 non-null  int64  
 6   Pressure_diff      145230 non-null  float64
 7   WindSpeed_mean     145230 non-null  float64
 8   WindSpeed_diff     145230 non-null  float64
 9   Humidity_mean      145230 non-null  float64
 10  Humidity_diff      145230 non-null  float64
 11  Cloud_mean         145230 non-null  float64
 12  Cloud_diff         145230 non-null  float64
 13  Temp_range         145230 non-null  float64
 14  Month              145230 non-null  int64  
 15  WindDir9am_angle   145230 non-null  float64
 16  Wi

In [5]:
# train test split
X = df.drop('RainTomorrow', axis = 1)
y = df['RainTomorrow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# Standardizing scale

scaler = StandardScaler()
X = scaler.fit_transform(X)

---

### Traning Logistic Regression

In [7]:
log_model = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000
)

log_model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [8]:
y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:,1]

In [9]:
print(confusion_matrix(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))

[[17628  5063]
 [ 1555  4800]]
              precision    recall  f1-score   support

           0       0.92      0.78      0.84     22691
           1       0.49      0.76      0.59      6355

    accuracy                           0.77     29046
   macro avg       0.70      0.77      0.72     29046
weighted avg       0.82      0.77      0.79     29046

ROC-AUC: 0.8499039901199229


---

## Traning Random Forest classifier

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [11]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [12]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

[[21826   865]
 [ 3545  2810]]
              precision    recall  f1-score   support

           0       0.86      0.96      0.91     22691
           1       0.76      0.44      0.56      6355

    accuracy                           0.85     29046
   macro avg       0.81      0.70      0.73     29046
weighted avg       0.84      0.85      0.83     29046

ROC-AUC: 0.8785077395797493


---

### Traning XGBClassifier

In [13]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight= (len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [14]:
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [15]:
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

[[18492  4199]
 [ 1446  4909]]
              precision    recall  f1-score   support

           0       0.93      0.81      0.87     22691
           1       0.54      0.77      0.63      6355

    accuracy                           0.81     29046
   macro avg       0.73      0.79      0.75     29046
weighted avg       0.84      0.81      0.82     29046

ROC-AUC: 0.8800772780801118


In [16]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
score = cross_val_score(xgb, X, y, cv=cv, scoring='roc_auc')

print("Mean AUC:", score.mean())
print("Std:", score.std())

Mean AUC: 0.8804745064824578
Std: 0.001496598014008604
